# Welcome to Bridging Research-Grade Models and Consumer Wearables: A Personalized Federated Learning Approach to Stress Prediction 

## Getting Started:

First, activate your virtual environment, if necessary, and download all necessary packages: 

In [ ]:
pip install -r requirements.txt

## Verify Installation:
Check that all core libraries are properly installed:

In [ ]:
# Test core imports
import gc
import sys

import numpy as np
import pandas as pd
import torch

print("All packages imported successfully!")
print(f"\nPython: {sys.version.split()[0]}")
print(f"PyTorch: {torch.version}")
print(f"NumPy: {np.version}")
print(f"Pandas: {pd.__version__}")
print(f"\nDevice: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Data Installation
### Option A: Automatic Download (Linux/MacOS/Bash present) <br>
If you're on Linux or Mac or Git Bash is available, run the cell below to automatically download and extract the WESAD dataset:

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

data_dir = Path("data/raw/wesad")

# If already present, skip
if data_dir.exists() and any(data_dir.iterdir()):
    print(f"✅ Dataset already present at: {data_dir.resolve()}")
else:
    # try bash if it exists (Linux/macOS/Git Bash/WSL)
    bash = shutil.which("bash")
    sh_script = Path("scripts/download_dataset.sh")

    if bash and sh_script.exists():
        subprocess.run([bash, str(sh_script)], check=True)
        print("Dataset downloaded via bash script!")
    else:
        print(" 'bash' not found (likely native Windows).")
        print("Proceeding with manual instructions.")

### Option B: Manual Download (All Platforms)

If automatic download doesn't work:

- Download: [WESAD Dataset](https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download)
- Extract the downloaded zip file
- Place the WESAD folder in: data/raw/wesad/

- Expected structure:

```
data/
└── raw/
    └── wesad/
        └── WESAD/
            ├── S2/
            ├── S3/
            └── ...

```

## Verify Dataset Location

Check if WESAD dataset is present: 

In [ ]:
from pathlib import Path

wesad_path = Path("data/raw/wesad/WESAD")

if wesad_path.exists():
    subjects = sorted(
        [d.name for d in wesad_path.iterdir() if d.is_dir() and d.name.startswith("S")]
    )
    print("WESAD dataset found!")
    print(f"Location: {wesad_path.absolute()}")
    print(f"Subjects ({len(subjects)}): {', '.join(subjects)}")
else:
    print("WESAD dataset NOT found!")
    print(f"Expected at: {wesad_path.absolute()}")
    print("\nPlease download and place the dataset in the correct location.")

## Configuration Setup
Set up your environment configuration:

In [ ]:
# Create .env file if it doesn't exist

from pathlib import Path

from dotenv import load_dotenv

env_file = Path(".env")

if not env_file.exists():
    env_example = Path(".env.example")

    if env_example.exists():
        with open(env_example) as f:
            content = f.read()

        with open(env_file, "w") as f:
            f.write(content)

        print("Created .env from .env.example")

    else:
        print("ℹ️ No .env.example found, using defaults")

else:
    print("ℹ️ .env already exists")

# Load environment variables
load_dotenv()
print("Configuration loaded")

You can now proceed with:

Data Preprocessing - Process raw WESAD signals
Feature Extraction - Extract time/frequency domain features
Model Training - Train baseline and federated models
Evaluation - Analyze results

Continue to the next section below to start preprocessing the data.

## Data Preprocessing

### Load and Merge WESAD Dataset

During this step, the biosensor data from the WESAD directory will be automatically extracted into several merged files in `.pkl` format.<br>

This may take a few minutes depending on the machine.

In [ ]:
from src.data import WESADResampleMerge

print("Initializing WESAD preprocessor...")
preprocessor = WESADResampleMerge(
    dataset_path="data/raw/wesad/WESAD/",
    IDs=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17],
)

# Loads metadata, merges wrist data (resample to BVP rate),chest data and filters labels and invalid temperatures
print("\nStarting preprocessing pipeline...")
preprocessor.resample_merge()

# Load the merged data
print("\nLoading merged datasets...")
chest_df = pd.read_pickle("data/raw/wesad/WESAD/merged/merged_chest.pkl")
wrist_df = pd.read_pickle("data/raw/wesad/WESAD/merged/merged_wrist.pkl")

# Display info
print("\n" + "=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)
print("\nChest Data:")
print(f"  Shape: {chest_df.shape}")
print(f"  Columns: {list(chest_df.columns)}")
print(f"  Memory: {chest_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nWrist Data:")
print(f"  Shape: {wrist_df.shape}")
print(f"  Columns: {list(wrist_df.columns)}")
print(f"  Memory: {wrist_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nLabel Distribution (Chest):")
print(chest_df["label"].value_counts().sort_index())

### Data Validation

In [ ]:
print("QUICK DATA VALIDATION CHECKS:\n")

#  Basic Info
print("\n1. CHEST DATA")
print(f"   Shape: {chest_df.shape}")
print(f"   Memory: {chest_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   Columns: {list(chest_df.columns)}")

print("\n2. WRIST DATA")
print(f"   Shape: {wrist_df.shape}")
print(f"   Memory: {wrist_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   Columns: {list(wrist_df.columns)}")

print("\n3. LABEL DISTRIBUTION")
print("\n   Chest:")
print(chest_df["label"].value_counts().sort_index())
print("\n   Wrist:")
print(wrist_df["label"].value_counts().sort_index())

# Check for Invalid Temps (discovered in EDA notebook 2)
print("\n4. TEMPERATURE VALIDATION")
chest_temp_col = "c_temp" if "c_temp" in chest_df.columns else "temp"
wrist_temp_col = "w_temp"

chest_invalid = (chest_df[chest_temp_col] <= 0).sum()
wrist_invalid = (wrist_df[wrist_temp_col] <= 0).sum()

print(f"   Chest temp ≤0: {chest_invalid} (should be 0)")
print(f"   Wrist temp ≤0: {wrist_invalid} (should be 0)")

if chest_invalid == 0 and wrist_invalid == 0:
    print(" Temperature interpolation successful!")
else:
    print(" Warning: Still have invalid temperatures")

# Check for Invalid eda values (discovered in EDA notebook 2)
print("\n5. EDA VALIDATION")
chest_eda_col = "c_eda" if "c_eda" in chest_df.columns else "eda"
wrist_eda_col = "w_eda"

chest_invalid = (chest_df[chest_eda_col] < 0).sum()
wrist_invalid = (wrist_df[wrist_eda_col] < 0).sum()

print(f"   Chest eda <0: {chest_invalid} (should be 0)")
print(f"   Wrist eda <0: {wrist_invalid} (should be 0)")

if chest_invalid == 0 and wrist_invalid == 0:
    print(" EDA interpolation successful! No negative artifacts.")
else:
    print(" Warning: Still have invalid eda artifacts")


print("\n6. MISSING DATA")
chest_nans = chest_df.isnull().sum()
wrist_nans = wrist_df.isnull().sum()

if chest_nans.sum() > 0:
    print(f"   Chest NaNs:\n{chest_nans[chest_nans > 0]}")
else:
    print("   Chest: No missing values")

if wrist_nans.sum() > 0:
    print(f"   Wrist NaNs:\n{wrist_nans[wrist_nans > 0]}")
else:
    print("   Wrist: No missing values")

print("\nData validation complete!")

## Signal Preprocessing 

We will preprocess each signal with their respective preprocessing steps, using NeuroKit2 for signals with peer-reviewed processing modules, and manual filters (Savitzky–Golay, FIR) for signals without standardised processing modules in the NK2 library.

### Chest Signals
*(All chest signals are at 700 Hz native sampling rate)*

#### ECG
- **Filtering**: 5th-order Butterworth high-pass at 0.5 Hz + 50 Hz notch filter (via NeuroKit2 `ecg_clean()`, "neurokit" method). Preserves full QRS morphology for downstream R-peak detection and HRV feature extraction.

#### EMG
- **Filtering**: 4th-order Butterworth high-pass at 100 Hz + constant detrending (via NeuroKit2 `emg_clean()`, "biosppy" method). The 100 Hz cutoff intentionally removes the lower EMG register to fully eliminate ECG contamination, leaving only muscle activation in the 100–500 Hz range.

#### EDA (Chest)
- **Filtering**: Butterworth low-pass at 3 Hz (via NeuroKit2 `eda_clean()`, "neurokit" method). At 700 Hz the 3 Hz cutoff is well within Nyquist, enabling proper tonic/phasic decomposition via `eda_phasic()` downstream.

#### TEMP (Chest)
- **Smoothing**: Savitzky–Golay filter with window size 11 and polynomial order 3. Temperature is a very slow-changing signal (changes over minutes) and no dedicated temperature module exists in NeuroKit2 or BioSPPy.

#### RESP
- **Filtering**: 2nd-order Butterworth band-pass at 0.05–3 Hz (via NeuroKit2 `rsp_clean()`, "khodadad2018" method). This preserves breathing rates from 3 to 180 breaths/min, which is preferable for stress detection over the tighter BioSPPy range (0.1–0.35 Hz ≈ 6–21 breaths/min), since stress-induced tachypnea can exceed 30 breaths/min (0.5 Hz).

#### ACC (Chest)
- **Smoothing**: Savitzky–Golay filter with window size 31 and polynomial order 5 (applied per axis: X, Y, Z). Removes high-frequency sensor noise while preserving body motion dynamics.

---

### Wrist Signals
*(All wrist signals are upsampled to 64 Hz in the resampling stage)*

#### BVP
- **Filtering**: Butterworth low-pass filter (via NeuroKit2 `ppg_clean()`, "elgendi" method, 64 Hz). BVP and PPG are physiologically identical signals, so the PPG pipeline is the correct processing family. The cleaned signal is suitable for downstream peak detection and HRV/IBI feature extraction.

#### EDA (Wrist)
- **Filtering**: Butterworth low-pass at 3 Hz (via NeuroKit2 `eda_clean()`, "neurokit" method, 64 Hz). At the native 4 Hz rate, Nyquist (2 Hz) sat below the 3 Hz cutoff, making any filter a no-op. After upsampling to 64 Hz (Nyquist = 32 Hz), the 3 Hz low-pass preserves all genuine EDA content while removing interpolation artifacts introduced by resampling.

#### TEMP (Wrist)
- **Smoothing**: Savitzky–Golay filter with polynomial order 3, window size scaled proportionally from the native-rate window (native 4 Hz → window 11, upsampled 64 Hz → window 176 + 1 = 177 to enforce odd-length requirement). Window scaling is necessary because Savitzky–Golay is sample-count-based rather than frequency-based.

#### ACC (Wrist)
- **Filtering**: FIR low-pass filter with 128 taps and a cutoff of 0.4 Hz normalised to Nyquist, applied via `filtfilt()` for zero-phase distortion (per axis: X, Y, Z). Designed to isolate slow orientation changes from hand tremor and high-frequency vibrations.

In [ ]:
# set config_files
CHEST_CONFIG = "config_files/dataset/wesad_chest_configuration.json"
WRIST_CONFIG = "config_files/dataset/wesad_wrist_configuration.json"

In [ ]:
from src.data import SignalPreprocessor

# preprocess the chest data
signal_preprocessor = SignalPreprocessor(
    "data/raw/wesad/WESAD/merged/merged_chest.pkl",
    "data/processed/wesad/chest_preprocessed.pkl",
    CHEST_CONFIG,
)
signal_preprocessor.preprocess_signals()

# preprocess the wrist data
signal_preprocessor = SignalPreprocessor(
    "data/raw/wesad/WESAD/merged/merged_wrist.pkl",
    "data/processed/wesad/wrist_preprocessed.pkl",
    WRIST_CONFIG,
    wrist=True,
)
signal_preprocessor.preprocess_signals()

## Data Windowing & Segmentation

Segments preprocessed signals into fixed-length windows for feature extraction.

Design decisions (justified by literature):

### Window size:
-   60s for all signals, Schmidt et al. (2018) WESAD benchmark.
-   This is sufficient for time-domain HRV, Poincaré, and LF/HF frequency-domain features.
-   VLF/ULF frequency-domain HRV features require ≥300s and are dropped (as in multiple WESAD studies the dataset conditions are too short for reliable VLF estimation).

### ACC sub-windows:
-   5s sub-windows within each 60s window, per Schmidt et al. (2018).
-   ACC is high-frequency motion data so 60s averaging washes out patterns.
-   Feature extraction will aggregate sub-window features (mean/std) into the 60s window.
 
### Slide / overlap:
-   Configurable. Schmidt benchmark uses 0.25s shift.
-  For RF, high overlap introduces correlated samples which according to literature is not ideal for tree-based models.
-   Therefore ablation is supported: 0.25s, 2s, 5s, 20s, 60s (no overlap).
 
### Label purity:
-   Windows spanning multiple labels are rejected. This avoids mixed label windows where part of the data is baseline and part is stress. We check this strictly.

### Subject isolation:
-   Windowing is always done per-subject. No window ever crosses subject boundaries.

In [ ]:
from src.windowing.wesad_windowing import WESADWindowing, summarize_ablation

# Paths
CHEST_PREPROCESSED = "data/processed/wesad/chest_preprocessed.pkl"
WRIST_PREPROCESSED = "data/processed/wesad/wrist_preprocessed.pkl"

CHEST_CONFIG = "config_files/dataset/wesad_chest_configuration.json"
WRIST_CONFIG = "config_files/dataset/wesad_wrist_configuration.json"

WINDOWED_OUTPUT_DIR = "data/processed/wesad/windowed/"

# Windowing parameters
WINDOW_SIZE_SEC = (
    60.0  # 60s window, sufficient for HRV time/freq/nonlinear features (Schmidt 2018)
)
ACC_SUBWINDOW_SEC = 5.0  # ACC sub-windowed at 5s per Schmidt et al. (2018)
SLIDE_SEC = 5.0  # changed from 0.25 for RF baseline, ablation below tests alternatives

# Chest windowing
print("\nCHEST DEVICE\n")

chest_windower = WESADWindowing(
    data_path=CHEST_PREPROCESSED,
    config_path=CHEST_CONFIG,
    output_dir=WINDOWED_OUTPUT_DIR,
    window_size_sec=WINDOW_SIZE_SEC,
    slide_sec=SLIDE_SEC,
    acc_subwindow_sec=ACC_SUBWINDOW_SEC,
    wrist=False,
)

chest_windower.load_data()
chest_windows = chest_windower.segment()
chest_windows_path, chest_meta_path = chest_windower.save_windows(chest_windows)

# Verify: materialise one window on demand to confirm the indexing works
sample = chest_windows[0]
sample_df = chest_windower.get_window_df(sample)
print(
    f"Sample window: sid: {sample['sid']}, label: {sample['label']}, "
    f"shape: {sample_df.shape}, is_overlapped: {sample['is_overlapped']}"
)

# ACC sub-windows extracted on demand example
acc_subs = chest_windower.extract_acc_subwindows(sample_df)
print(f"ACC sub-windows: {len(acc_subs)} × {acc_subs[0].shape if acc_subs else 'N/A'}")

# Wrist windowing
print("\nWRIST DEVICE\n")

wrist_windower = WESADWindowing(
    data_path=WRIST_PREPROCESSED,
    config_path=WRIST_CONFIG,
    output_dir=WINDOWED_OUTPUT_DIR,
    window_size_sec=WINDOW_SIZE_SEC,
    slide_sec=SLIDE_SEC,
    acc_subwindow_sec=ACC_SUBWINDOW_SEC,
    wrist=True,
)

wrist_windower.load_data()
wrist_windows = wrist_windower.segment()
wrist_windows_path, wrist_meta_path = wrist_windower.save_windows(wrist_windows)

# verify
sample_w = wrist_windows[0]
sample_w_df = wrist_windower.get_window_df(sample_w)
print(
    f"Sample window: sid: {sample_w['sid']}, label: {sample_w['label']}, "
    f"shape: {sample_w_df.shape}, is_overlapped: {sample_w['is_overlapped']}"
)
acc_subs_w = wrist_windower.extract_acc_subwindows(sample_w_df)
print(
    f"ACC sub-windows: {len(acc_subs_w)} × {acc_subs_w[0].shape if acc_subs_w else 'N/A'}"
)

# Ablation study (RF slide sensitivity)
# For the Random Forest, high overlap introduces correlated training samples, according to literature.
# We test five values to identify the best trade-off between sample count
# and inter-sample independence. The benchmark 0.25s is included.
#
# is_overlapped flag ensures validation sets only ever use independent windows

print("\nABLATION — CHEST\n")

chest_ablation_results = chest_windower.run_ablation(
    slide_values=[0.25, 2.0, 5.0, 20.0, 60.0]
)
summarize_ablation(chest_ablation_results, window_size_sec=WINDOW_SIZE_SEC)

# Release chest data before wrist ablation to avoid holding both in memory
del chest_windower.df
gc.collect()

print("\nABLATION — WRIST\n")

wrist_ablation_results = wrist_windower.run_ablation(
    slide_values=[0.25, 2.0, 5.0, 20.0, 60.0]
)
summarize_ablation(wrist_ablation_results, window_size_sec=WINDOW_SIZE_SEC)

## Manual Feature Extraction

Extracts handcrafted features from each windowed segment for the Random Forest baseline model.

**Strategy per sensor:**
- **ECG** (700 Hz): HRV time-domain (RMSSD, SDNN, pNN50), frequency-domain (LF, HF — VLF/ULF dropped, require ≥300s), nonlinear (SD1/SD2, SampEn), wave morphology (P/Q/S/T), ECG-derived respiration (EDR)
- **EMG** (700 Hz): Statistical features + spectral band energy (100–350 Hz, above the 100 Hz high-pass cutoff applied in preprocessing)
- **EDA** (700 Hz chest / 64 Hz wrist): tonic/phasic decomposition via cvxEDA (SCL mean/std, SCR peak count and amplitude)
- **BVP** (64 Hz): PPG-derived HRV time, frequency, and nonlinear features — primary bridge signal to the Fitbit
- **RESP** (700 Hz): breathing rate, inhalation/exhalation durations, I/E ratio
- **TEMP** (both): mean, std, min, max, range, slope
- **ACC** (both): features on 5s sub-windows aggregated (mean/std) per Schmidt et al. (2018)

**Post-extraction:**
- Per-subject min-max normalization using baseline condition (label 1.0) as reference - follows Schmidt (2018) and prevents inter-subject leakage
- Median imputation for NaN values produced by NeuroKit2 on short/irregular windows

In [ ]:
import gc

from src.ml_pipeline.feature_extraction import run_feature_pipeline_streaming

# Output paths
CHEST_FEATURES_OUT = "data/processed/wesad/features/chest_features.pkl"
WRIST_FEATURES_OUT = "data/processed/wesad/features/wrist_features.pkl"

# Paths to the preprocessed merged pickles (NOT loaded into RAM)
CHEST_DATA_PATH = "data/processed/wesad/chest_preprocessed.pkl"
WRIST_DATA_PATH = "data/processed/wesad/wrist_preprocessed.pkl"

# Chest:
chest_features = run_feature_pipeline_streaming(
    data_path=CHEST_DATA_PATH,
    windows=chest_windows,
    config_path=CHEST_CONFIG,
    output_path=CHEST_FEATURES_OUT,
    wrist=False,
    baseline_label=1.0,
    normalize=True,
    impute=True,
    impute_strategy="median",
)
gc.collect()

# Wrist:
wrist_features = run_feature_pipeline_streaming(
    data_path=WRIST_DATA_PATH,
    windows=wrist_windows,
    config_path=WRIST_CONFIG,
    output_path=WRIST_FEATURES_OUT,
    wrist=True,
    baseline_label=1.0,
    normalize=True,
    impute=True,
    impute_strategy="median",
)
gc.collect()

### Verification

In [ ]:
import pandas as pd

CHEST_FEATURES_OUT = "data/processed/wesad/features/chest_features.pkl"
WRIST_FEATURES_OUT = "data/processed/wesad/features/wrist_features.pkl"

chest_features = pd.read_pickle(CHEST_FEATURES_OUT)
wrist_features = pd.read_pickle(WRIST_FEATURES_OUT)

meta_cols = ["sid", "label", "window_idx", "is_overlapped"]
for name, df in [("CHEST", chest_features), ("WRIST", wrist_features)]:
    feat_cols = [c for c in df.columns if c not in meta_cols]
    print(f"\n{name} FEATURE MATRIX")
    print(f"  Shape                : {df.shape}")
    print(f"  Feature count        : {len(feat_cols)}")
    print(f"  Subjects             : {sorted(df['sid'].unique())}")
    print(
        f"  Label distribution   : {df['label'].value_counts().sort_index().to_dict()}"
    )
    print(f"  NaN count            : {df[feat_cols].isna().sum().sum()}")
    print(
        f"  Inf count            : {(df[feat_cols].abs() == float('inf')).sum().sum()}"
    )
    print(f"  Independent windows  : {(~df['is_overlapped']).sum()}")
    print(f"  Overlapped windows   : {df['is_overlapped'].sum()}")
    print("\n  Feature value ranges (first 10 features):")
    print(
        df[feat_cols].describe().T[["mean", "std", "min", "max"]].head(10).to_string()
    )

print("\nSave/load integrity check passed.")

### TODO: 
- write the actual notebook cells for feature selection and LOSOCV RF
- one more eda notebook for visualization - ablations, feature correlation, tsne, metrics etc
- REMEMBER TO SEE HOW YOU WILL HANDLE CLASS IMBALANCE
- do tsne
- for the NN should I use encoder for fe or is manual fe with nk2 fine
- these two sources have really good descriptions of methodology, follow https://link.springer.com/article/10.1007/s12652-022-04365-z https://ieeexplore.ieee.org/document/9995340 https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=9669993
- remember to do correlation analysis on the features extracted from eacgh ablation using different windows

IN THE FEATURE SELECTION NOTEBOOK!!! use this from the feature selection . py self._importances = pd.Series(self._rf.feature_importances_, index=feat_cols).sort_values(ascending=False)
Store importance scores for later inspection. I can plot the top 20 features and discuss which physiological signals are most predictive.
and in KBEsT self._scores = pd.Series(self._selector.scores_, index=feat_cols).sort_values(ascending=False)
review: does my current code support selectively training base model on wrist and chest OR wrist alone? for FL i will need to push the model that is on wrist only and see how that affects accuracy
